# Домашнее задание по теме «Spark SQL»  
Задание:



1.   Выберите 15 стран с наибольшим процентом переболевших на 31 марта (в выходящем датасете необходимы колонки: iso_code, страна, процент переболевших)
2.   Top 10 стран с максимальным зафиксированным кол-вом новых случаев за последнюю неделю марта 2021 в отсортированном порядке по убыванию
(в выходящем датасете необходимы колонки: число, страна, кол-во новых случаев)
3. Посчитайте изменение случаев относительно предыдущего дня в России за последнюю неделю марта 2021. (например: в россии вчера было 9150 , сегодня 8763, итог: -387) (в выходящем датасете необходимы колонки: число, кол-во новых случаев вчера, кол-во новых случаев сегодня, дельта)  

Формат выполнения работы: загрузите скрипт и результат выборки на гитхаб и пришлите ссылку на выполненную работу для проверки экспертом

Инструменты для выполнения работы: Apache Spark

In [150]:
# Установка pyspark
!pip install --quiet pyspark

In [151]:
# Установка
!pip install -q black blackcellmagic

# магическая для форматирования кода
%load_ext blackcellmagic

The blackcellmagic extension is already loaded. To reload it, use:
  %reload_ext blackcellmagic


In [152]:
from pyspark.sql import SparkSession
from pyspark import SparkFiles
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("CovidAnalysis").getOrCreate()

# Чтение из файла

In [153]:
# чтение из файла
covid_data_file_url = "https://raw.githubusercontent.com/pl8242/spark_sql-homework/refs/heads/main/covid-data.csv"
spark.sparkContext.addFile(covid_data_file_url)
file_path = "file://" + SparkFiles.get("covid-data.csv")
df = spark.read.option("inferSchema", "true").option("header", "true").csv(file_path)

In [154]:
# выводим список и типы данных
df.printSchema()

root
 |-- iso_code: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- location: string (nullable = true)
 |-- date: date (nullable = true)
 |-- total_cases: double (nullable = true)
 |-- new_cases: double (nullable = true)
 |-- new_cases_smoothed: double (nullable = true)
 |-- total_deaths: double (nullable = true)
 |-- new_deaths: double (nullable = true)
 |-- new_deaths_smoothed: double (nullable = true)
 |-- total_cases_per_million: double (nullable = true)
 |-- new_cases_per_million: double (nullable = true)
 |-- new_cases_smoothed_per_million: double (nullable = true)
 |-- total_deaths_per_million: double (nullable = true)
 |-- new_deaths_per_million: double (nullable = true)
 |-- new_deaths_smoothed_per_million: double (nullable = true)
 |-- reproduction_rate: double (nullable = true)
 |-- icu_patients: double (nullable = true)
 |-- icu_patients_per_million: double (nullable = true)
 |-- hosp_patients: double (nullable = true)
 |-- hosp_patients_per_million: 

In [155]:
# выводим всю таблицу на просмотр
# df.show()

1. **Выберите 15 стран с наибольшим процентом переболевших на 31 марта (в выходящем датасете необходимы колонки: iso_code, страна, процент переболевших)**

In [156]:
# в задании не указано за какой год на 31 марта вывести статитистику
# проверим за сколько лет данные
df_year = df.selectExpr("YEAR(`date`) AS year").distinct().orderBy("year").show()

+----+
|year|
+----+
|2020|
|2021|
+----+



In [157]:
# в п.2-3 задания - март 2021, аналогично сделаем для этого пункта
df_max_cases_share = (
    df.select(
        "iso_code",
        "location",
        (F.col("total_cases_per_million") / 10000).alias("cases_share"),
    )
    .where(F.col("date") == ("2021-03-31"))
    .sort(F.col("total_cases_per_million").desc())
    .limit(15)
)
df_max_cases_share.show()

+--------+-------------+------------------+
|iso_code|     location|       cases_share|
+--------+-------------+------------------+
|     AND|      Andorra|        15.5439073|
|     MNE|   Montenegro|14.523725399999998|
|     CZE|      Czechia|        14.3088484|
|     SMR|   San Marino|        13.9371796|
|     SVN|     Slovenia|10.370805800000001|
|     LUX|   Luxembourg|         9.8473424|
|     ISR|       Israel|          9.625106|
|     USA|United States|          9.203011|
|     SRB|       Serbia|         8.8263286|
|     BHR|      Bahrain|         8.4888601|
|     PAN|       Panama|         8.2287391|
|     PRT|     Portugal|         8.0586997|
|     EST|      Estonia|         8.0226816|
|     SWE|       Sweden|         7.9697443|
|     LTU|    Lithuania|         7.9388647|
+--------+-------------+------------------+



2. **Top 10 стран с максимальным зафиксированным кол-вом новых случаев за последнюю неделю марта 2021 в отсортированном порядке по убыванию (в выходящем датасете необходимы колонки: число, страна, кол-во новых случаев)**

In [158]:
# фильтруем по периоду и тоталу (исключаем строки с Null в continent)
df_max_new_cases = (
    df.filter(
        F.col("date").between("2021-03-25", "2021-03-31")
        & F.col("continent").isNotNull()
    )
    .groupBy("location")
    .agg(
        F.max_by("date", "new_cases").alias("date"),
        F.max("new_cases").alias("max_new_cases"),
    )
    .orderBy(F.col("max_new_cases").desc())
    .limit(10)
    .select("date", "location", "max_new_cases")
)
df_max_new_cases.show()

+----------+-------------+-------------+
|      date|     location|max_new_cases|
+----------+-------------+-------------+
|2021-03-25|       Brazil|     100158.0|
|2021-03-26|United States|      77321.0|
|2021-03-31|        India|      72330.0|
|2021-03-31|       France|      59054.0|
|2021-03-31|       Turkey|      39302.0|
|2021-03-26|       Poland|      35145.0|
|2021-03-31|      Germany|      25014.0|
|2021-03-26|        Italy|      24076.0|
|2021-03-25|         Peru|      19206.0|
|2021-03-26|      Ukraine|      18226.0|
+----------+-------------+-------------+



3. **Посчитайте изменение случаев относительно предыдущего дня в России за последнюю неделю марта 2021. (например: в россии вчера было 9150 , сегодня 8763, итог: -387) (в выходящем датасете необходимы колонки: число, кол-во новых случаев вчера, кол-во новых случаев сегодня, дельта)**

In [159]:
# импортируем Window для оконной функции, применяем lag
from pyspark.sql.window import Window

df_delta_cases_rus = (
    df.filter(
        (F.col("date").between("2021-03-24", "2021-03-31"))
        & (F.col("location") == "Russia")
    )
    .withColumn(
        "prev_new_cases",
        F.lag("new_cases", 1).over(Window.orderBy("date")),
    )
    .select(
        "date",
        "prev_new_cases",
        "new_cases",
        (F.col("new_cases") - F.col("prev_new_cases")).alias("delta_new_cases"),
    )
    .where(F.col("delta_new_cases").isNotNull())
)
df_delta_cases_rus.show()

+----------+--------------+---------+---------------+
|      date|prev_new_cases|new_cases|delta_new_cases|
+----------+--------------+---------+---------------+
|2021-03-25|        8769.0|   9128.0|          359.0|
|2021-03-26|        9128.0|   9073.0|          -55.0|
|2021-03-27|        9073.0|   8783.0|         -290.0|
|2021-03-28|        8783.0|   8979.0|          196.0|
|2021-03-29|        8979.0|   8589.0|         -390.0|
|2021-03-30|        8589.0|   8162.0|         -427.0|
|2021-03-31|        8162.0|   8156.0|           -6.0|
+----------+--------------+---------+---------------+

